# Eval Pipeline (no constitution)

Builds an **evaluation** dataset: standalone content-moderation inputs → model outputs →
jailbreak augmentation → merged dataset. No constitution (that's the *training* pipeline).

Everything is keyed off a single **`data_dir`** working root; `Datasets/` and `Data_cache/`
live under it. Models default to their registry **role** (pass `None`). `resume=True` (the
default) continues an interrupted run; `resume=False` restarts a stage clean.

In [ ]:
from redact import generate_inputs, generate_outputs, generate_jailbreaks, build_dataset

DATA_DIR = "./runs/eval"      # one working root for the whole run
MODEL = None                  # None -> uncensored_gen role default (venice-uncensored)
NUM_CATEGORIES = 3
SAMPLES_PER_CATEGORY = 15

## Step 1 — Inputs (standalone, meta-prompt)

In [ ]:
inputs = generate_inputs(
    data_dir=DATA_DIR,
    samples_per_category=SAMPLES_PER_CATEGORY,
    num_categories=NUM_CATEGORIES,
    model=MODEL,
)
print(f"{len(inputs)} accepted inputs")
inputs.head()

## Step 2 — Outputs (model responses + checker)

In [ ]:
outputs = generate_outputs(data_dir=DATA_DIR, model=MODEL, max_per_category=20)
print(f"{len(outputs)} responses")
outputs.head()

## Step 3 — Jailbreak augmentation

`include_translation=False` drops the costly DeepSeek translation family (recommended first
pass). For multi-round escalation, pass `settings_per_iteration=default_escalation_schedule()`.

In [ ]:
jailbreaks = generate_jailbreaks(
    data_dir=DATA_DIR,
    model=MODEL,
    include_translation=False,
    entry_types=["harmful"],
)
print(f"{len(jailbreaks)} jailbreak rows")
jailbreaks["technique"].value_counts().head(10)

## Step 4 — Merge into the eval dataset

In [ ]:
dataset = build_dataset(data_dir=DATA_DIR)
dataset["dataset_type"].value_counts()

## Alternative — one-shot, config-driven

The same run as a single call via a recipe + input-params. Each stage writes a
`{stage}.run.json` manifest under `{data_dir}/Datasets/` (inspect with
`python scripts/inspect_run.py ./runs/eval`).

In [ ]:
from redact import run_pipeline

summary = run_pipeline(
    {
        "dataset_type": "eval",
        "data_dir": DATA_DIR,
        "stages": ["inputs", "outputs", "jailbreaks", "build"],
        "augmentations": {"include_translation": False},
    },
    params={
        "inputs": {"samples_per_category": SAMPLES_PER_CATEGORY, "num_categories": NUM_CATEGORIES},
        "outputs": {"max_per_category": 20},
        "jailbreaks": {"entry_types": ["harmful"]},
    },
)
summary